This is the preprocessing notebook, in this notebook, we devided this process into 5 phases

Phase 1: Data Exploration and Structural cleaning

Our dataset is not "simple", it is a bit complicated that is why this step is going to be essential.

in this step we convert the time in the main dataset to seconds, which will be easier for the machine to read it.
(converting everything to the same unit : s)

In [1]:
import pandas as pd
import numpy as np

# --- PHASE 1: DATA EXPLORATION & STRUCTURAL CLEANING ---
# Our dataset is a bit complicated, so this step is essential to make sure 
# the machine doesn't "cheat" and the numbers are in the right format.

# 1. Load the raw data
# We are using the 2024-2026 dataset
df = pd.read_excel('F1_dataset_2024-2026.xlsx')

# 2. Prevent Data Leakage (CRITICAL STEP)
# We must drop columns that are only known AFTER the race happens.
# If we keep 'Points' or 'Race Status', the model will already know the answer!
leaky_cols = [
    'Points', 'Laps', 'Race Status', 
    'Fastest Lap Rank', 'Fastest Lap Time', 'Fastest Lap Speed'
]
df = df.drop(columns=leaky_cols)

# 3. Date Formatting
# We need this as a datetime object so we can split the years properly later
df['Date'] = pd.to_datetime(df['Date'])


# ADD HERE 👇
df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], errors='coerce')
df['Age'] = 2026 - df['Date of Birth'].dt.year
df = df.drop(columns=['Date of Birth'])
# 4. Feature Transformation: Time to Seconds
# Qualifying times are strings like "1:30.031". We convert them to floats 
# so the computer can actually do math with them.
def convert_time_to_seconds(time_val):
    # Handle the '\N' or empty gaps found in F1 data
    if pd.isna(time_val) or time_val == '\\N' or time_val == '':
        return np.nan
    
    try:
        if ':' in str(time_val):
            minutes, seconds = str(time_val).split(':')
            return int(minutes) * 60 + float(seconds)
        return float(time_val)
    except Exception:
        return np.nan

# Applying the conversion to the qualifying rounds
quali_columns = ['Q1 Time', 'Q2 Time', 'Q3 Time']
for col in quali_columns:
    df[col] = df[col].apply(convert_time_to_seconds)

# 5. Fixing the Grid Position
# Sometimes Grid Position is missing (22 cases in our data).
# Usually, this means a Pit Lane start, so we put them at the back (Position 20).
df['Grid Position'] = pd.to_numeric(df['Grid Position'], errors='coerce')
df['Grid Position'] = df['Grid Position'].fillna(20)

# 6. Cleaning Categorical Junk
# We have too many nationalities. 'Driver' and 'Constructor' already tell us who is who.
# Dropping redundant columns to keep the feature count low for Phase 4.
redundant_cols = ['Driver Nationality', 'Constructor Nationality', 'Country', 'Circuit Location']
df = df.drop(columns=redundant_cols)

# Final Check
print("Updated Data Types:\n", df.dtypes)
print("\nRemaining Nulls (Should only be in Q2/Q3 for eliminated players):\n", df.isnull().sum())

# Previewing our clean, non-leaky data
df.head()

Updated Data Types:
 Year                       int64
Round                      int64
Race Name                    str
Date              datetime64[us]
Circuit                      str
Latitude                 float64
Longitude                float64
Altitude (m)               int64
Driver                       str
Constructor                  str
Grid Position            float64
Quali Position           float64
Q1 Time                  float64
Q2 Time                  float64
Q3 Time                  float64
Final Position             int64
Age                        int32
dtype: object

Remaining Nulls (Should only be in Q2/Q3 for eliminated players):
 Year                0
Round               0
Race Name           0
Date                0
Circuit             0
Latitude            0
Longitude           0
Altitude (m)        0
Driver              0
Constructor         0
Grid Position       0
Quali Position      1
Q1 Time            15
Q2 Time           254
Q3 Time           502
Final 

,Year,Round,Race Name,Date,Circuit,Latitude,Longitude,Altitude (m),Driver,Constructor,Grid Position,Quali Position,Q1 Time,Q2 Time,Q3 Time,Final Position,Age
0,2024,1,Bahrain Grand Prix,2024-03-02,Bahrain International Circuit,26.0325,50.5106,7,Max Verstappen,Red Bull,1.0,1.0,90.031,89.374,89.179,1,29
1,2024,1,Bahrain Grand Prix,2024-03-02,Bahrain International Circuit,26.0325,50.5106,7,Sergio Pérez,Red Bull,5.0,5.0,90.221,89.932,89.537,2,36
2,2024,1,Bahrain Grand Prix,2024-03-02,Bahrain International Circuit,26.0325,50.5106,7,Carlos Sainz,Ferrari,4.0,4.0,89.909,89.573,89.507,3,32
3,2024,1,Bahrain Grand Prix,2024-03-02,Bahrain International Circuit,26.0325,50.5106,7,Charles Leclerc,Ferrari,2.0,2.0,90.243,89.165,89.407,4,29
4,2024,1,Bahrain Grand Prix,2024-03-02,Bahrain International Circuit,26.0325,50.5106,7,George Russell,Mercedes,3.0,3.0,90.350,89.922,89.485,5,28


Phase 2: Splitting

we need to split our data before performing calculations to prevent DATA LEAKAGE.

In [2]:
# --- PHASE 2: TEMPORAL SPLIT ---
# Goal: We need to split the data by Year to simulate a real-world scenario.
# We train on 2024, test our accuracy on 2025, and keep 2026 as a "Blind" prediction.

# 1. Isolate the "Future" (2026) immediately
# This data stays in a vault; we don't touch it until the very end.
df_2026 = df[df['Year'] == 2026].copy()

# 2. Create the working dataset for training and testing (2024-2025)
df_model = df[df['Year'] != 2026].copy()

# 3. Define the Target and the "Eraser" list
# We drop 'Year' and 'Date' because they are just markers—not actual performance features.
# We also drop 'Race Name' and 'Circuit' here if we want to rely strictly on numerical stats, 
# but for now, we only drop the essentials to keep the structure clean.
target = 'Final Position'
drop_cols = [target, 'Year', 'Date', 'Race Name','Round']

# 4. Split Train (2024) and Test (2025)
# This mimics a real season-to-season transition.
X_train = df_model[df_model['Year'] == 2024].drop(columns=drop_cols)
y_train = df_model[df_model['Year'] == 2024][target]

X_test = df_model[df_model['Year'] == 2025].drop(columns=drop_cols)
y_test = df_model[df_model['Year'] == 2025][target]

# 5. Prepare the Prediction Features for the 2026 season
# We don't have 'y_predict' because those races haven't happened yet in our simulation!
X_predict = df_2026.drop(columns=drop_cols)

print(f"Training features shape (2024): {X_train.shape}")
print(f"Testing features shape (2025):  {X_test.shape}")
print(f"Prediction features shape (2026): {X_predict.shape}")

Training features shape (2024): (479, 12)
Testing features shape (2025):  (479, 12)
Prediction features shape (2026): (22, 12)


phase 3: Handeling Nulls

it is basically imputation, We filled the "holes" (null values) in your data so the computer doesn't crash.

The "Normal" stuff: For things like race speed, we used the Median (the middle value) to fill gaps.

The "Eliminated" stuff: For qualifying times (Q2 and Q3), we used 999. This tells the computer: "This driver didn't make it to this round; they are at the very back.

Q2 and Q3 by natural are empty because the players got eliminated.

In [3]:
# --- PHASE 3: IMPUTATION (Handling the Holes) ---
# We need to fill the null values so the computer doesn't crash.
# But we have to be smart: a missing Q3 time means the driver was too slow to make it there!

# 1. Handling Qualifying Times (The "Elimination" Penalty)
# Instead of using a random big number like 999, we find the slowest lap 
# in our 2024 training data and add a 10-second penalty. 
# This tells the model: "This driver didn't qualify, they are at the very back."
slowest_q1 = X_train['Q1 Time'].max()
realistic_penalty = slowest_q1 + 10

# We apply this penalty to all qualifying columns
quali_cols = ['Q1 Time', 'Q2 Time', 'Q3 Time']
for col in quali_cols:
    X_train[col] = X_train[col].fillna(realistic_penalty)
    X_test[col] = X_test[col].fillna(realistic_penalty)
    X_predict[col] = X_predict[col].fillna(realistic_penalty)

# 2. Handling Ranking & Position (Median Imputation)
# For things like Quali Position, we use the Median (the middle value).
# This is a safe "normal" value to fill any remaining gaps.
rank_cols = ['Quali Position', 'Grid Position']

for col in rank_cols:
    # IMPORTANT: We calculate the median from the 2024 Train set ONLY 
    # to avoid leaking information from the future (2025/2026).
    col_median = X_train[col].median()
    
    X_train[col] = X_train[col].fillna(col_median)
    X_test[col] = X_test[col].fillna(col_median)
    X_predict[col] = X_predict[col].fillna(col_median)

# 3. Final Null Audit
# Double-checking to make sure every single "hole" is filled.
print("Final Null Audit (Should all be 0):")
print(X_train[quali_cols + rank_cols].isnull().sum())
print(f"\nRealistic Penalty Value used for eliminated drivers: {realistic_penalty:.2f} seconds")

Final Null Audit (Should all be 0):
Q1 Time           0
Q2 Time           0
Q3 Time           0
Quali Position    0
Grid Position     0
dtype: int64

Realistic Penalty Value used for eliminated drivers: 127.78 seconds


Phase 4: Categorical Encoding 

in this section we are going to use the following: 
* one hot encoding, rare label encoding and ordinal encoding 

In [4]:
from sklearn.preprocessing import OneHotEncoder

# --- PHASE 4: CATEGORICAL ENCODING ---
# We can't feed names like "Ferrari" into a regression model.
# We need to turn these into numbers without creating a fake "ranking" order.

# 1. Identify the columns we want to encode
# We focus on the most important categories that define the race.
nominal_cols = ['Driver', 'Constructor', 'Circuit']

# 2. Rare Label Encoding (The "Clean-up" Step)
# If a driver or circuit only appears once (maybe a replacement driver), 
# it can make the model unstable. We group these into a "Rare_Label" category.
def handle_rare_labels(df_train, df_test, df_predict, columns, threshold=0.01):
    for col in columns:
        # We find the labels that appear in less than 1% of the training data
        counts = df_train[col].value_counts(normalize=True)
        rare_labels = counts[counts < threshold].index
        
        # We replace them across all sets to keep everything consistent
        df_train[col] = df_train[col].replace(rare_labels, 'Rare_Label')
        df_test[col] = df_test[col].replace(rare_labels, 'Rare_Label')
        df_predict[col] = df_predict[col].replace(rare_labels, 'Rare_Label')
    return df_train, df_test, df_predict

X_train, X_test, X_predict = handle_rare_labels(X_train, X_test, X_predict, nominal_cols)

# 3. One-Hot Encoding
# This creates a new column for every Driver/Team with a 1 or 0.
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# We fit the encoder ONLY on the 2024 Train set (The Golden Rule)
ohe.fit(X_train[nominal_cols])

def apply_ohe_safe(df_set, encoder, columns):
    # Transform the text columns into a matrix of 1s and 0s
    encoded_array = encoder.transform(df_set[columns])
    
    # Create a new DataFrame with clear column names (e.g., 'Driver_Max Verstappen')
    encoded_df = pd.DataFrame(
        encoded_array, 
        columns=encoder.get_feature_names_out(columns), 
        index=df_set.index
    )
    
    # Drop the original text columns and attach the new 1/0 columns
    return pd.concat([df_set.drop(columns=columns), encoded_df], axis=1)

# Apply the transformation to all our sets
X_train = apply_ohe_safe(X_train, ohe, nominal_cols)
X_test = apply_ohe_safe(X_test, ohe, nominal_cols)
X_predict = apply_ohe_safe(X_predict, ohe, nominal_cols)

print(f"Encoding Finished!")
print(f"New feature count: {X_train.shape[1]} columns")

Encoding Finished!
New feature count: 66 columns


Phase 5 is Feature Scaling 

in this phase we have a major "size problem" so we are going to fix that :if we don't scale, it will think the Speed is 230 times more important than the Grid Position simply because the number is bigger

Standardization is usually the safest and most professional choice. However, because F1 has occasional outliers (crashes or slow pit stops), Robust Scaling is the "Pro" move.

However we will use here the standar way 

In [5]:
from sklearn.preprocessing import StandardScaler

# --- PHASE 5: FEATURE SCALING (The Fixed Version) ---
# We need to scale our numbers, but we have to make sure we don't 
# call columns we dropped in Phase 1 (like Fastest Lap Speed).

# 1. Define only the columns that actually exist in our X_train now
# These are the ones that survived the Phase 1 cleanup.
available_numeric = [
    'Q1 Time', 'Q2 Time', 'Q3 Time', 'Quali Position', 'Grid Position'
]

# 2. Safety Check: Filter the list just in case
numeric_features = [col for col in available_numeric if col in X_train.columns]

# 3. Initialize and Fit
scaler = StandardScaler()
scaler.fit(X_train[numeric_features])

# 4. Transform
X_train[numeric_features] = scaler.transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])
X_predict[numeric_features] = scaler.transform(X_predict[numeric_features])

print(f"Successfully scaled: {numeric_features}")

Successfully scaled: ['Q1 Time', 'Q2 Time', 'Q3 Time', 'Quali Position', 'Grid Position']


In [6]:
# --- FINAL EXPORT (The "Excel Maker") ---
# Now that we've finished Phase 5, we save the results.
# This part ensures we don't have to run the preprocessing every time we train a model.

X_train.to_excel('X_train_preprocessed.xlsx', index=False)
X_test.to_excel('X_test_preprocessed.xlsx', index=False)
X_predict.to_excel('X_predict_preprocessed.xlsx', index=False)

y_train.to_excel('y_train.xlsx', index=False)
y_test.to_excel('y_test.xlsx', index=False)

print("Preprocessing finished. Excel files are ready for your models!")

Preprocessing finished. Excel files are ready for your models!


In [7]:
X_train.isnull().sum().sum()

np.int64(0)

In [8]:
X_train.isnull().sum().sum()

np.int64(0)

In [9]:
import joblib

joblib.dump(scaler, "scaler.pkl")
joblib.dump(ohe, "ohe.pkl")

print("✅ scaler.pkl and ohe.pkl saved!")

✅ scaler.pkl and ohe.pkl saved!
